# NB_patient_360
## Silver → Gold |  Health Network

**Target Table:** `adf_healthgroup_metadatadriven.gold.patient_360`  
**Grain:** One row per patient  
**Load Type:** Full rebuild — `CREATE OR REPLACE TABLE … AS SELECT`  
**Source:** STTM `sttm_silver_to_gold.xlsx`, Sheet `patient_360`

### Source Tables
| Table | Role |
|---|---|
| `adf_healthgroup_metadatadriven.silver.patients` | Driving table |
| `adf_healthgroup_metadatadriven.silver.hospitals` | LEFT JOIN on `registered_hospital_id = hospital_id` |
| `adf_healthgroup_metadatadriven.silver.insurance_providers` | LEFT JOIN on `insurance_provider_id = insurance_provider_id` |
| `adf_healthgroup_metadatadriven.silver.lab_results` | Aggregated sub-query grouped by `patient_id` |

### Notes
- `date_of_birth` is stored as `TIMESTAMP` in the silver layer; cast to `DATE` before computing age.
- Age is computed as `FLOOR(DATEDIFF(CURRENT_DATE(), CAST(date_of_birth AS DATE)) / 365.25)`.
- `total_tests`, `abnormal_test_count`, and `critical_test_count` default to `0` when no lab results exist.
- `last_test_date` is `NULL` when no lab results exist.

In [0]:
# Set the Unity Catalog catalog name used throughout this notebook
catalog = "adf_healthgroup_metadatadriven"
print(f"Using catalog: {catalog}")

In [0]:
%sql
-- ============================================================
-- NB_patient_360
-- Target: adf_healthgroup_metadatadriven.gold.patient_360
-- Grain : One row per patient
-- Load  : Full rebuild (CREATE OR REPLACE)
-- ============================================================

CREATE OR REPLACE TABLE adf_healthgroup_metadatadriven.gold.patient_360 AS

WITH lab_agg AS (
    -- Aggregate lab results to patient level
    SELECT
        patient_id,
        COUNT(*)                                             AS total_tests,
        COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END) AS abnormal_test_count,
        COUNT(CASE WHEN result_status = 'Critical' THEN 1 END) AS critical_test_count,
        MAX(test_date)                                       AS last_test_date
    FROM adf_healthgroup_metadatadriven.silver.lab_results
    GROUP BY patient_id
)

SELECT
    -- Patient core
    p.patient_id,
    p.first_name,
    p.last_name,
    CAST(p.date_of_birth AS DATE)                                   AS date_of_birth,
    FLOOR(
        DATEDIFF(CURRENT_DATE(), CAST(p.date_of_birth AS DATE)) / 365.25
    )::INT                                                           AS age,
    p.gender,
    p.city,
    p.state,

    -- Hospital (via LEFT JOIN)
    p.registered_hospital_id                                        AS hospital_id,
    h.hospital_name,

    -- Insurance (via LEFT JOIN)
    p.insurance_provider_id,
    ip.provider_name                                                AS insurance_provider_name,

    -- Lab aggregates (default to 0 / NULL when no results)
    COALESCE(la.total_tests,          0)                            AS total_tests,
    COALESCE(la.abnormal_test_count,  0)                            AS abnormal_test_count,
    COALESCE(la.critical_test_count,  0)                            AS critical_test_count,
    la.last_test_date

FROM adf_healthgroup_metadatadriven.silver.patients            AS p
LEFT JOIN adf_healthgroup_metadatadriven.silver.hospitals      AS h
    ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adf_healthgroup_metadatadriven.silver.insurance_providers AS ip
    ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg                                              AS la
    ON p.patient_id = la.patient_id

ORDER BY p.patient_id

In [0]:
%sql
-- ============================================================
-- Post-load validation
-- ============================================================

-- Row count and sanity checks
SELECT
    COUNT(*)                                        AS total_rows,
    COUNT(DISTINCT patient_id)                      AS distinct_patients,
    SUM(CASE WHEN patient_id IS NULL THEN 1 END)    AS null_pk_count,
    MIN(age)                                        AS min_age,
    MAX(age)                                        AS max_age,
    SUM(CASE WHEN total_tests < 0 THEN 1 END)       AS negative_test_count,
    SUM(CASE WHEN hospital_name IS NULL THEN 1 END) AS missing_hospital_name,
    SUM(CASE WHEN insurance_provider_name IS NULL THEN 1 END) AS missing_insurance_name
FROM adf_healthgroup_metadatadriven.gold.patient_360

In [0]:
updated_df = spark.table("adf_healthgroup_metadatadriven.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))